<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/sefrqo_plus_core_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# [CELL 1 - Environment Setup & Imports]
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style="whitegrid")

print("SEFRQO-Plus Simulation Environment Initialized.")

SEFRQO-Plus Simulation Environment Initialized.


In [2]:
# [CELL 2 - Query Structure-Aware Encodings]
import re

def anonymize_sql_query(sql_text: str) -> str:
    """
    Anonymizes SQL text by replacing schema-specific identifiers
    (table names and aliases) with structural placeholders (R1, R2, etc.).
    """
    # Simplified mock implementation for structural anonymization demonstration
    tables = re.findall(r'FROM\s+([a-zA-Z0-9_]+)|JOIN\s+([a-zA-Z0-9_]+)', sql_text, re.IGNORECASE)
    flat_tables = [t[0] or t[1] for t in tables if t[0] or t[1]]

    anon_sql = sql_text
    for idx, tbl in enumerate(set(flat_tables), start=1):
        anon_sql = re.sub(r'\b' + tbl + r'\b', f'R{idx}', anon_sql)

    return anon_sql

def extract_ast_structural_signatures(sql_text: str) -> np.ndarray:
    """
    Extracts coarse-grained structural signatures from the query AST
    (number of joins, tables, filters) to form normalized structural vectors.
    """
    num_joins = upper_sql = sql_text.upper().count("JOIN")
    num_tables = sql_text.upper().count("FROM") + upper_sql
    num_filters = sql_text.upper().count("WHERE") + sql_text.upper().count("AND")

    raw_signature = np.array([num_joins, num_tables, num_filters], dtype=float)
    # L2 Normalization
    norm = np.linalg.norm(raw_signature)
    if norm > 0:
        return raw_signature / norm
    return raw_signature

sample_sql = "SELECT * FROM company_name AS cn JOIN movie_companies AS mc ON cn.id = mc.company_id WHERE cn.country_code = 'ru';"
print("Original SQL:", sample_sql)
print("Anonymized SQL:", anonymize_sql_query(sample_sql))
print("Normalized AST Structural Vector:", extract_ast_structural_signatures(sample_sql))

Original SQL: SELECT * FROM company_name AS cn JOIN movie_companies AS mc ON cn.id = mc.company_id WHERE cn.country_code = 'ru';
Anonymized SQL: SELECT * FROM R2 AS cn JOIN R1 AS mc ON cn.id = mc.company_id WHERE cn.country_code = 'ru';
Normalized AST Structural Vector: [0.40824829 0.81649658 0.40824829]


In [3]:
# [CELL 3 - Latency-Based Reward Model R_Latency]

def compute_latency_reward(baseline_latency: float, actual_latency: float) -> float:
    """
    Computes the database-aware latency reward using R_Latency = tanh(ln(delta_t)),
    where delta_t = t_d / t_i is the latency improvement ratio.
    """
    delta_t = baseline_latency / actual_latency
    # R_Latency formulation from SEFRQO-Plus
    reward = np.tanh(np.log(delta_t))
    return float(reward), float(delta_t)

# Test with various speedups and slowdowns
baseline = 1000.0 # ms
test_latencies = [500.0, 1000.0, 2000.0, 250.0] # 2x speedup, baseline, 2x slowdown, 4x speedup

print("\n--- Q_GRPO Latency Reward Evaluation ---")
for t_i in test_latencies:
    rew, d_t = compute_latency_reward(baseline, t_i)
    print(f"Baseline: {baseline}ms | Actual: {t_i}ms | Ratio (delta_t): {d_t:.2f} | Reward: {rew:.3f}")


--- Q_GRPO Latency Reward Evaluation ---
Baseline: 1000.0ms | Actual: 500.0ms | Ratio (delta_t): 2.00 | Reward: 0.600
Baseline: 1000.0ms | Actual: 1000.0ms | Ratio (delta_t): 1.00 | Reward: 0.000
Baseline: 1000.0ms | Actual: 2000.0ms | Ratio (delta_t): 0.50 | Reward: -0.600
Baseline: 1000.0ms | Actual: 250.0ms | Ratio (delta_t): 4.00 | Reward: 0.882


In [4]:
# [CELL 4 - Group Relative Advantage Computation]

def compute_group_relative_advantages(rewards: list) -> list:
    """
    Computes group relative advantages A_hat for a group of G generated hint outputs
    based on their reward distribution.
    """
    r_arr = np.array(rewards)
    mean_r = np.mean(r_arr)
    std_r = np.std(r_arr)

    # Avoid zero division
    if std_r == 0.0:
        std_r = 1e-8

    advantages = (r_arr - mean_r) / std_r
    return advantages.tolist()

# Simulate a group of 4 hint outputs for a query
group_rewards = [0.65, 0.82, -0.15, 0.45]
advantages = compute_group_relative_advantages(group_rewards)

print("\n--- Group Relative Advantage Feedback ---")
for idx, (r, adv) in enumerate(zip(group_rewards, advantages), start=1):
    print(f"Hint Output {idx}: Reward = {r:+.2f} -> Advantage (A_hat) = {adv:+.3f}")
print(f"Theoretical Max Advantage Bound for G=4: |A_hat| <= sqrt(4-1) = {np.sqrt(3):.3f}")


--- Group Relative Advantage Feedback ---
Hint Output 1: Reward = +0.65 -> Advantage (A_hat) = +0.566
Hint Output 2: Reward = +0.82 -> Advantage (A_hat) = +1.031
Hint Output 3: Reward = -0.15 -> Advantage (A_hat) = -1.618
Hint Output 4: Reward = +0.45 -> Advantage (A_hat) = +0.020
Theoretical Max Advantage Bound for G=4: |A_hat| <= sqrt(4-1) = 1.732


In [5]:
# [CELL 5 - RAG Dynamic Prompt Constructor with Multi-Dimensional Feedback]

def construct_sefrqo_plus_dynamic_prompt(query_sql: str, stats: dict, best_plan: str, best_gain: float, recent_plan: str, recent_gain: float, reference_pairs: list) -> str:
    """
    Constructs the dynamic prompt incorporating system prompt, k similar reference pairs,
    query statistics, historical best plan (with gain eta*), and most recent plan (with gain eta).
    """
    sp = (
        "## System Prompt\n"
        "You are a database expert performing the query optimization task. "
        "Generate a valid and efficient query hint (full-plan or join-order mode) "
        "to minimize execution latency.\n"
    )

    refs_text = "## Here are the similar query-answer pairs you can reference:\n"
    for idx, (ref_q, ref_p) in enumerate(reference_pairs, 1):
        refs_text += f"Ref {idx} SQL: {ref_q}\nRef {idx} Plan Hint: {ref_p}\n"

    stats_text = f"## Here is the query and its corresponding statistics:\nSQL: {query_sql}\nTable Cardinalities: {stats}\n"

    feedback_text = (
        f"## The best historical query plan you generated and its performance gain:\n"
        f"Plan: {best_plan}\nPerformance Gain (eta*): {best_gain * 100:.1f}%\n\n"
        f"## The most recent query plan you generated and its performance gain:\n"
        f"Plan: {recent_plan}\nPerformance Gain (eta): {recent_gain * 100:.1f}%\n"
    )

    constraints = (
        "## Please generate a better query plan than provided references.\n"
        "Do not repeat failure patterns, stick to the correct syntax format, and optimize join orders."
    )

    dynamic_prompt = f"{sp}\n{refs_text}\n{stats_text}\n{feedback_text}\n{constraints}"
    return dynamic_prompt

# Example usage
sample_stats = {"table_cn": 1500, "table_mc": 45000, "filter_selectivity": 0.05}
refs = [("SELECT * FROM R1 JOIN R2 ON R1.id = R2.id", "Leading((R1 R2))")]
prompt_output = construct_sefrqo_plus_dynamic_prompt(
    query_sql=sample_sql,
    stats=sample_stats,
    best_plan="NestedLoop(cn mc)",
    best_gain=0.45,
    recent_plan="SeqScan(cn) HashJoin(mc)",
    recent_gain=-0.12, # negative feedback from a suboptimal recent trial
    reference_pairs=refs
)

print("\n--- Generated SEFRQO-Plus Dynamic Prompt ---")
print(prompt_output[:600] + "\n[... truncated for display ...]")


--- Generated SEFRQO-Plus Dynamic Prompt ---
## System Prompt
You are a database expert performing the query optimization task. Generate a valid and efficient query hint (full-plan or join-order mode) to minimize execution latency.

## Here are the similar query-answer pairs you can reference:
Ref 1 SQL: SELECT * FROM R1 JOIN R2 ON R1.id = R2.id
Ref 1 Plan Hint: Leading((R1 R2))

## Here is the query and its corresponding statistics:
SQL: SELECT * FROM company_name AS cn JOIN movie_companies AS mc ON cn.id = mc.company_id WHERE cn.country_code = 'ru';
Table Cardinalities: {'table_cn': 1500, 'table_mc': 45000, 'filter_selectivity': 0.05}

[... truncated for display ...]
